# Concept

While tracking manual variables with Autograd works for simple algorithms, building deep architectures with millions of parameters requires structural organization. PyTorch provides this via the torch.nn.Module base class.

Every neural network model you build from scratch must inherit from this parent class. It automatically registers and tracks all structural weights and biases within its layers, allowing you to easily view, move, or pass parameters directly to production optimizers.

In [1]:
import torch
import torch.nn as nn

class CustomLinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.rand(1, requires_grad=True))
        self.bias = nn.Parameter(torch.rand(1, requires_grad=True))

    def forward(self, x):
        return x*self.weights + self.bias

model = CustomLinearModel()

# Look how easily we can inspect all active tracking parameters inside!
for name, param in model.named_parameters():
    print(f"Parameter registered: {name} | Shape: {param.shape}")

Parameter registered: weights | Shape: torch.Size([1])
Parameter registered: bias | Shape: torch.Size([1])


# Exercise

Build a model architecture class named Perceptron that inherits from nn.Module.

    Inside its constructor (__init__), call the parent class initialization step. Then, instantiate a single weight tracking parameter named w initialized to [0.5] and a bias parameter named b initialized to [0.0]. Wrap both parameter initializations within nn.Parameter() to register them to the model hierarchy.

    Define its forward(self, x) execution function to compute and return x⋅w+b.

In [3]:
class Perceptron(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.tensor([0.5], requires_grad=True))
        self.b = nn.Parameter(torch.tensor([0.0], requires_grad=True))

    def forward(self, x):
        return x*self.w + self.b

# Concept

Manually configuring every tracking parameter matrix with nn.Parameter() becomes chaotic when designing deep networks containing dozens of wide layers. To automate this process, PyTorch provides pre-packaged structural layers within the torch.nn module.

The most fundamental building block is nn.Linear(in_features, out_features). This automatically instantiates, manages, and executes an optimized weight matrix W and bias vector b underneath the hood, executing a standard affine transformation:
y=xWT+b

In [4]:
class StandardMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # nn.Linear automatically instantiates weights and biases for you
        # Expects an input vector of size 4; outputs a transformed vector of size 2
        self.fc1 = nn.Linear(in_features=4, out_features=2)
        
    def forward(self, x):
        # Passing x directly through the structural layer object
        return self.fc1(x)

# Create a mock batch of data: 3 samples, each containing 4 feature elements
mock_batch = torch.randn(3, 4)

model = StandardMLP()
output = model(mock_batch)
print(output.shape) # Output: torch.Size([3, 2])

torch.Size([3, 2])


# Exercise

Build a model class named DeepClassifier that inherits from nn.Module.

In its constructor, define two separate structural layers:

        self.layer1: An nn.Linear layer that takes 8 input features and transforms them into 4 hidden features.

        self.layer2: An nn.Linear layer that takes those 4 hidden features and outputs a single 1 output feature.

In the forward(self, x) method, pass the incoming data tensor x through layer1 to create an intermediate representation, and then pass that intermediate result directly through layer2. Return the final result.

In [5]:
class DeepClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(in_features=8, out_features=4)
        self.layer2 = nn.Linear(in_features=4, out_features=1)
    def forward(self, x):
        return self.layer2(self.layer1(x))

# Concept

If you pass data through a linear layer, and then immediately pass it through another linear layer, your deep network is still just computing a single, flat linear equation. Mathematically, chaining linear transformations together always collapses into a single linear transformation:

W2​(W1​x+b1​)+b2​=(W2​W1​)x+(W2​b1​+b2​)=Wcombined​x+bcombined​

To prevent your layers from collapsing and to allow your model to learn complex, non-linear patterns (like curves, intersections, or boundaries), you must inject a non-linear activation function between your linear layers.

The most common activation function in deep learning is ReLU (Rectified Linear Unit), which simply filters out negative numbers:
ReLU(x)=max(0,x)

In [6]:
class NonLinearMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(8, 4)
        # Instantiate the ReLU activation module
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(4, 1)
        
    def forward(self, x):
        # We pass the output of fc1 through relu BEFORE it enters fc2
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Exercise

Modify your previous architecture. Create a model class named DeepClassifier that implements an 8-input feature system, transforms it to a 4-feature hidden layer, applies a ReLU activation, and then passes the result to a final 1-feature output layer. Correct your keyword argument names to match PyTorch's native parameters (in_features, out_features).

In [7]:
class DeepClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(in_features=8, out_features=4)
        self.layer2 = nn.Linear(in_features=4, out_features=1)
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))

# Concept

When your models grow to include dozens of sequential layers, tracking and declaring every single layer and activation function as a separate line of code inside __init__ becomes incredibly tedious.

To clean this up, PyTorch provides nn.Sequential. This container acts as a structural wrapper that chains modules together in a predefined order. When data passes into an nn.Sequential block, it automatically routes the output of one layer directly into the input of the next layer sequentially, shrinking your entire forward method down into a clean, single-line operation.

In [8]:
class CompactMLP(nn.Module):
    def __init__(self):
        super().__init__()
        
        # nn.Sequential groups the entire sub-graph layout into one object
        self.network_pipeline = nn.Sequential(
            nn.Linear(in_features=8, out_features=4),
            nn.ReLU(),
            nn.Linear(in_features=4, out_features=2),
            nn.ReLU(),
            nn.Linear(in_features=2, out_features=1)
        )
        
    def forward(self, x):
        # The entire chain executes automatically with a single call
        return self.network_pipeline(x)

# Exercise

Refactor your previous DeepClassifier model to use nn.Sequential. Name the container object self.main_structure. Ensure it handles the 8-input features to 4-hidden features transformation, passes through a ReLU activation, and finishes with a 1-feature output layer. Implement the corresponding forward pass to route data through that container.

In [9]:
class DeepClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network_pipline = nn.Sequential(nn.Linear(in_features=8, out_features=4),
                                            nn.ReLU(),
                                            nn.Linear(in_features=4, out_features=1)
                                            )
        
    def forward(self, x):
        return self.network_pipline(x)

# FINAL EXERCISE

Objective:
Build a standard Multi-Layer Perceptron (MLP) named FunctionalMLP from scratch using everything you have learned in this module.

Requirements:
1. Inheritance: Your class must inherit from nn.Module and properly initialize its base class elements.
2. Architecture Layout: Inside your constructor, define a single nn.Sequential block assigned to the exact variable name self.network_block. Your block must map out this specific structural sequence:
Input Layer: A linear layer that takes 16 input features and scales them to 8 hidden features.
Activation 1: A nn.ReLU() activation block.
Hidden Layer: A linear layer that takes those 8 hidden features and scales them down to 4 hidden features.
Activation 2: Another nn.ReLU() activation block.
Output Layer: A final linear layer that takes those 4 hidden features and transforms them into 2 distinct output features (representing class logits).
3. Execution Routing: Implement the forward(self, x) pass to feed an incoming data batch directly through your self.network_block container.
4. Verification Track: Outside your class block, write a verification script that:
Instantiates your model: model = FunctionalMLP().
Creates a mock batch of synthetic data: mock_batch = torch.randn(5, 16). This simulates a batch size of 5 samples, where each sample contains 16 raw features.
Passes that mock batch through your model: predictions = model(mock_batch).
Prints the final shape property (.shape) of your predictions tensor.

In [10]:
class FunctionalMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network_block = nn.Sequential(nn.Linear(16, 8),
                                          nn.ReLU(),
                                          nn.Linear(8, 4),
                                          nn.ReLU(),
                                          nn.Linear(4, 2)
                                          )
    def forward(self, x):
        return self.network_block(x)

In [11]:
model = FunctionalMLP()
mock_batch = torch.randn(5, 16)

predictions = model(mock_batch)
print(predictions.shape)

torch.Size([5, 2])
